In [1]:
# this script attempts to reproduce the baseline PSO optimiser from Feiyang

import numpy as np

import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
# imports from src folder
from src.params import ENV_CONFIG, HFSS_FILE
from src.utils import load_hfss_channel
from src.env import NearFieldBeamFocusingEnv
from src.pso_optimizer import run_pso_focus_optimization


In [3]:
Hsi, selected_freq = load_hfss_channel(
    HFSS_FILE,
    target_freq=ENV_CONFIG["fc"]
)

print(f"Selected frequency: {selected_freq / 1e9:.4f} GHz")
print(f"Full Hsi shape: {Hsi.shape}")

Selected frequency: 3.5000 GHz
Full Hsi shape: (24, 24)


In [4]:
env = NearFieldBeamFocusingEnv(
    Hsi=Hsi,
    **ENV_CONFIG
)



In [5]:
case = {
    "r_dl_user": 3.0,
    "phi_dl_user": 20.0,
    "theta_dl_user": 0.0,

    "r_ul_user": 4.0,
    "phi_ul_user": -25.0,
    "theta_ul_user": 0.0,

    "seed": 5003,
}

In [6]:
result = run_pso_focus_optimization(
    env,
    case,
    seed=case["seed"]
)

In [7]:
print("\n--- PSO result ---")

print(f"Reference SI:    {result['si_ref_db']:.3f} dB")
print(f"Best SI:         {result['best_si_db']:.3f} dB")
print(f"SI improvement:  {result['si_improvement_db']:.3f} dB")

print("\nBest Tx focus:")
print(result["best_tx_focus"])

print("\nBest Rx focus:")
print(result["best_rx_focus"])

print("\nGain constraints:")
print(f"DL gain loss: {result['dl_gain_loss_db']:.3f} dB")
print(f"UL gain loss: {result['ul_gain_loss_db']:.3f} dB")
print(f"Feasible:     {result['is_gain_feasible']}")

print("\nSelection mode:")
print(result["selection_mode"])


--- PSO result ---
Reference SI:    -35.074 dB
Best SI:         -57.965 dB
SI improvement:  22.891 dB

Best Tx focus:
{'r': 8.0, 'phi': -45.984773686328296, 'theta': 7.05018895123914}

Best Rx focus:
{'r': 2.0731984401411387, 'phi': -23.400939060823127, 'theta': -10.683348252295268}

Gain constraints:
DL gain loss: 1.756 dB
UL gain loss: 2.999 dB
Feasible:     True

Selection mode:
gain-feasible best SI
